In [1]:
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KernelDensity
from shapely.geometry import Point
from shapely.geometry import Polygon
from shapely.ops import unary_union
from scipy.ndimage import maximum_filter
import folium
from branca.colormap import linear
import tempfile
import webbrowser
import os


In [2]:
trees = gpd.read_file("/Users/prince/philly-tree-mapper/data/processed/philly_trees_located.geojson").to_crs(4326)


In [4]:
# 6 trees as representatives
# 2 trees w hella acerifolia and rubrum
# 2 trees w mid zie quercus bicolo & sccharinum
# 2 trees with few opacas
acerifolias = trees.loc[trees["scientific_name"] == "Platanus acerifolia"].to_crs(epsg=2272)
acer_rubrums = trees.loc[trees["scientific_name"] == "Acer rubrum"].to_crs(epsg=2272)
quercus_bicolors = trees.loc[trees["scientific_name"] == "Quercus bicolor"].to_crs(epsg=2272)
acer_saccharinums = trees.loc[trees["scientific_name"] == "Acer saccharinum"].to_crs(epsg=2272)
pinus_nigras = trees.loc[trees["scientific_name"] == "Pinus nigra"].to_crs(epsg=2272)
ilex_opacas = trees.loc[trees["scientific_name"] == "Ilex opaca"].to_crs(epsg=2272)

In [5]:
def show_in_browser(m):
    """Open a folium map in the default browser without saving to project dir."""
    tmp = tempfile.NamedTemporaryFile(suffix=".html", delete=False)
    tmp.close()
    m.save(tmp.name)
    webbrowser.open(f"file://{tmp.name}")

In [9]:
def density_folium(gdf, pad=5280, num_cells=200, hood_size=10, bandwidth=3000, kernel="gaussian"):
    # gdf is assumed to be in EPSG:2272 already
    coords = np.array([(geom.x, geom.y) for geom in gdf.geometry])
    
    kde = KernelDensity(kernel=kernel, bandwidth=bandwidth)
    kde.fit(coords)
    
    x_min, y_min, x_max, y_max = gdf.total_bounds
    xx, yy = np.meshgrid(
        np.linspace(x_min - pad, x_max + pad, num_cells),
        np.linspace(y_min - pad, y_max + pad, num_cells),
    )
    grid_coords = np.column_stack([xx.ravel(), yy.ravel()])
    log_density = kde.score_samples(grid_coords)
    density = np.exp(log_density).reshape(xx.shape)
    
    contour_threshold = density.mean() + 2 * density.std()
    cs = plt.contour(xx, yy, density, levels=[contour_threshold])
    polygons = []
    for path_collection in cs.allsegs:
        for segment in path_collection:
            if len(segment) >= 4 and not np.allclose(segment[0], segment[-1], atol=1e-6):
                segment = np.vstack([segment, segment[0]])
            if len(segment) >= 4:
                poly = Polygon(segment)
                if poly.is_valid and poly.area > 0:
                    polygons.append(poly)
    plt.close()
    
    clusters_gdf = gpd.GeoDataFrame(
        {"cluster_id": range(len(polygons))},
        geometry=polygons,
        crs="EPSG:2272",
    )
    joined = gpd.sjoin(gdf, clusters_gdf, how="inner", predicate="within")
    counts = joined.groupby("cluster_id").size().rename("point_count")
    clusters_gdf = clusters_gdf.join(counts, on="cluster_id")
    clusters_gdf["point_count"] = clusters_gdf["point_count"].fillna(0).astype(int)
    
    clusters_wgs84 = clusters_gdf.to_crs(epsg=4326)
    points_wgs84 = gdf.to_crs(epsg=4326)
    
    map_center = [points_wgs84.geometry.y.mean(), points_wgs84.geometry.x.mean()]
    m = folium.Map(location=map_center, zoom_start=13, tiles="cartodbpositron")
    
    colormap = linear.YlOrRd_09.scale(
        clusters_wgs84["point_count"].min(),
        clusters_wgs84["point_count"].max(),
    )
    colormap.caption = "Points per cluster"
    
    folium.GeoJson(
        clusters_wgs84,
        style_function=lambda feature: {
            "fillColor": colormap(feature["properties"]["point_count"]),
            "color": "black",
            "weight": 1,
            "fillOpacity": 0.8,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["cluster_id", "point_count"],
            aliases=["Cluster:", "Points:"],
        ),
    ).add_to(m)
    
    for _, row in points_wgs84.iterrows():
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=.1,
            color="navy",
            fill=True,
            fill_opacity=0.4,
        ).add_to(m)
    
    colormap.add_to(m)
    m.save("clusters_map.html")
    return m

In [10]:
m = density_folium(ilex_opacas, bandwidth=500, num_cells=1000)
show_in_browser(m)